# Setup and Orientation

**Applied ML in Production · Session 0**

---

This module is about the distance between a model that works and a model that
is *used*. That distance is where most machine learning projects die, and
almost none of it is algorithmic.

Over five weeks we will take one problem — predicting default on SME loan
applications — from a vague business complaint all the way to a running
prediction service. The same dataset, the same problem, ten sessions. Nothing
gets thrown away and restarted.

This session does two things: it confirms your environment works, and it
introduces the data you will be living with.

## How to work through this

Run every code cell (`Shift + Enter`), read the output, *then* read the
commentary. Cells build on each other, so if something errors, run from the top.

Each notebook ends with exercises and a short "if you remember nothing else"
summary. The exercises are where the learning actually happens — the code cells
are just the demonstration.

## Learning objectives

After this session you will be able to:

- Set up and verify the module's Python environment.
- Load the spine dataset and describe what each column means.
- State the business problem this module solves, in one sentence.
- Explain why the module uses a single dataset throughout rather than a new one
  each week.

## Environment

From the repository root, once:

```bash
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python -r requirements-aimodule.txt
```

Then select `.venv` as this notebook's kernel. The next cell checks that
everything the module needs is present.

In [1]:
import sys
import importlib

REQUIRED = [
    "numpy", "pandas", "matplotlib", "seaborn", "sklearn", "imblearn",
    "mlflow", "fastapi", "streamlit", "joblib",
]

print(f"Python {sys.version.split()[0]}")
if sys.version_info[:2] != (3, 12):
    print("  WARNING: this module is built and tested on Python 3.12")

missing = []
for name in REQUIRED:
    try:
        module = importlib.import_module(name)
        print(f"  ok   {name:<12} {getattr(module, '__version__', '')}")
    except ImportError:
        missing.append(name)
        print(f"  MISSING  {name}")

if missing:
    print(f"\nInstall the missing packages before continuing: {', '.join(missing)}")
else:
    print("\nEnvironment ready.")

Python 3.12.13
  ok   numpy        2.1.3


  ok   pandas       2.2.3
  ok   matplotlib   3.9.2


  ok   seaborn      0.13.2
  ok   sklearn      1.5.2


  ok   imblearn     0.12.4


  ok   mlflow       2.17.2
  ok   fastapi      0.115.5
  ok   streamlit    1.40.2
  ok   joblib       1.4.2

Environment ready.


## The problem

A lender issues working-capital loans to small and medium businesses across
Nepal. Credit officers assess each application by hand. Volume has grown faster
than the credit team, decisions have become inconsistent between branches, and
the default rate has drifted upward.

The request that arrives on your desk is: *"can we use machine learning to
predict which loans will default?"*

That sentence is not yet a machine learning problem. Turning it into one — with
a defined target, a defined population, a defined decision, and a defined
measure of success — is the entire content of session 1.

In [2]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/loan_default.csv")

loans = pd.read_csv(DATA_PATH, parse_dates=["application_date"])

print(f"{len(loans):,} applications, {loans.shape[1]} columns")
print(f"{loans['application_date'].min():%Y-%m-%d} to {loans['application_date'].max():%Y-%m-%d}")
loans.head()

12,000 applications, 15 columns
2023-01-01 to 2024-12-31


,loan_id,application_date,branch,sector,employment_type,loan_amount,tenure_months,interest_rate,annual_income,credit_score,has_collateral,previous_loans,days_past_due_history,recovery_agent_assigned,defaulted
0,NL200000,2023-01-01,Birgunj-01,Service,Business Owner,417000.0,36,10.28,194000.0,652.0,Yes,2,40,Yes,1
1,NL200001,2023-01-01,Biratnagar-03,Service,Self-Employed,154000.0,12,9.82,NaN,695.0,Yes,1,0,No,0
2,NL200002,2023-01-01,Kathmandu-03,Trade,Salaried,163000.0,24,13.04,948000.0,565.0,No,2,22,No,0
3,NL200003,2023-01-01,Lalitpur-01,Tourism,Salaried,634000.0,18,12.98,327000.0,505.0,Yes,2,21,No,0
4,NL200004,2023-01-01,Birgunj-04,Trade,Salaried,196000.0,60,10.08,235000.0,643.0,Yes,3,2,No,0


In [3]:
print("Column types and completeness\n")
summary = pd.DataFrame({
    "dtype": loans.dtypes.astype(str),
    "missing": loans.isna().sum(),
    "missing_pct": (loans.isna().mean() * 100).round(1),
    "unique": loans.nunique(),
})
print(summary.to_string())

print(f"\nDefault rate: {loans['defaulted'].mean():.1%}")

Column types and completeness

                                  dtype  missing  missing_pct  unique
loan_id                          object        0          0.0   12000
application_date         datetime64[ns]        0          0.0     731
branch                           object        0          0.0      40
sector                           object        0          0.0       6
employment_type                  object        0          0.0       4
loan_amount                     float64        0          0.0    1362
tenure_months                     int64        0          0.0       7
interest_rate                   float64        0          0.0     778
annual_income                   float64     1122          9.4    1613
credit_score                    float64      851          7.1     493
has_collateral                   object        0          0.0       2
previous_loans                    int64        0          0.0       9
days_past_due_history             int64        0          0

### Three things that output already told you

**The classes are imbalanced.** Roughly one loan in eight defaults. A model that
predicts "never defaults" for every application is right about 88 percent of the
time and completely useless. Accuracy is already a trap, and we have not written
a line of modelling code.

**Two columns have missing values.** `annual_income` and `credit_score`. The
convenient move is `dropna()`. Session 2 shows what that quietly costs you.

**One column is not what it appears to be.** We will not say which one yet.
Session 2 is about finding it.

Do not fix any of this now. Each problem is the subject of a later session, and
seeing it bite before seeing it fixed is the point.

## The ten sessions

| Week | Session | Topic |
|------|---------|-------|
| 1 | 1 | ML problem framing and the industry workflow |
| 1 | 2 | Data preparation — and the leakage hunt |
| 2 | 3 | Feature engineering and class imbalance |
| 2 | 4 | Building models with scikit-learn |
| 3 | 5 | Evaluation metrics and the cost of being wrong |
| 3 | 6 | Tuning and error analysis |
| 4 | 7 | Pipelines and experiment tracking |
| 4 | 8 | Introduction to MLOps |
| 5 | 9 | Deployment and APIs |
| 5 | 10 | End-to-end project |

Sessions 4, 5 and 6 revisit ground DCS 404 covered. They are not a repeat — the
question there was *how does this algorithm work*, and the question here is
*how do I choose, judge, and defend one*. Where a derivation is needed, the
notebook links back to the DCS 404 page rather than repeating it.

## If you remember nothing else

One problem, ten sessions, one dataset. The messiness you just saw in the data
summary is not an accident and it is not something to clean up before the real
work starts — it *is* the real work.

## Your turn

1. Open `data/README.md` and read the column descriptions. Which column would
   you be suspicious of, and why?
2. Compute the default rate separately for each `sector`. Which sector looks
   riskiest?
3. Write one sentence stating what decision this model would actually change.
   Keep it — session 1 opens with it.